<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Prompt Engineering for Unsupervised Audio Classification Using CLAP</b></h1>
</div>

## Theoretical Foundations

### Technical Context

Zero-shot audio classification with CLAP replaces a task-specific learned classifier by a comparison between audio and natural-language representations. The decision boundary is therefore influenced by both the pretrained representation and the textual form used to describe each candidate class.

### Core Audio–Text Model

For an audio clip $x$ and candidate prompt $p_c$,

$$
a=f_{audio}(x),
\qquad
t_c=f_{text}(p_c).
$$

After normalization, classification is performed by ranking cross-modal similarities between $a$ and all $t_c$.

### Notation and Conventions

- $x$ — ESC-50 audio sample;
- $c$ — ground-truth class;
- $p_c$ — textual prompt associated with class $c$;
- $a$ — audio embedding;
- $t_c$ — text embedding;
- $s_c$ — audio-text similarity score;
- $\hat c$ — predicted Top-1 class;
- $A$ — aggregate accuracy;
- $A_k$ — Top-k accuracy.

All reported accuracies refer to the fixed ESC-50 evaluation population used in the implementation.

### Analytical Scope

The theory covers zero-shot inference, prompt-conditioned text prototypes, normalized embedding similarity, class-wise and Top-k evaluation, correlation analysis, and limits of external benchmark comparison. It does not cover CLAP pretraining, ESC-50 fine-tuning, learned prompt optimization, or supervised classifier training.

## 1. Data and Output Paths

### Reproducibility Role

Path handling is not part of the classifier mathematics, but it is part of the experimental design. A result is reproducible only when the same input resources and generated artifacts can be located without machine-specific assumptions.

Repository-relative paths separate the experiment from a particular workstation while keeping data, figures, and tables traceable to one project root.


## 2. ESC-50 Dataset Preparation

### Dataset Definition

ESC-50 contains 2,000 five-second environmental recordings organized into 50 semantic classes. The benchmark is balanced, with 40 clips per class.

In this project, ESC-50 acts strictly as an evaluation set. Its labels define the candidate semantic vocabulary, but its audio examples are not used to fit a classifier or update CLAP parameters.

### Zero-Shot Constraint

The pretrained model must remain unchanged. Downloading or preparing the dataset changes only data availability, not the learned representation.


## 3. Dataset Exploration

### Evaluation Population

Before inference, the experiment must establish the exact sample population and candidate class set. A change in class ordering, spelling, or label normalization can alter the generated text prompts and therefore the resulting embeddings.

The sorted class list becomes the canonical index that links text-embedding rows, similarity scores, predicted indices, and human-readable labels.


## 4. CLAP Model Initialization

### Contrastive Audio-Language Representation

CLAP learns two encoders that map audio and text into a shared latent space. During inference,

$$
a = f_{audio}(x),
\qquad
t = f_{text}(p),
$$

where $x$ is an audio clip and $p$ is a text prompt.

Evaluation mode preserves the pretrained parameters and disables training-specific behavior. Using one fixed checkpoint across all strategies is essential because otherwise prompt effects would be confounded with representation changes.


## 5. Prompt Strategy Definition

### Prompt Formulation as an Experimental Variable

A prompt template supplies linguistic context around the same class label. For class $c$, a template $P$ produces text

$$
p_c = P(c).
$$

Although two templates may be semantically similar, the text encoder can map them to different points in embedding space. Prompt engineering therefore changes the decision geometry without changing model weights.

### Controlled Comparison

Only the template wording may change. Class labels, model, audio inputs, normalization, and scoring must remain fixed.


## 6. Zero-Shot Classification Framework

### Embedding Normalization

Let $a$ be the audio embedding and $t_c$ the text embedding for candidate class $c$. L2 normalization gives

$$
\tilde a = \frac{a}{\|a\|_2},
\qquad
\tilde t_c = \frac{t_c}{\|t_c\|_2}.
$$

The dot product then equals cosine similarity:

$$
s_c = \tilde a^T\tilde t_c
    = \cos(\theta_c).
$$

### Prediction Rule

The Top-1 prediction is

$$
\hat c=
\operatorname*{arg\,max}_c s_c.
$$

Sorting all $s_c$ values produces the Top-k ranking. No learned classifier head is required because the text embeddings themselves define the candidate decision prototypes.


## 7. Full Prompt Benchmark

### Fair Benchmarking

Each prompt strategy is evaluated on the same 2,000 samples and the same 50 candidate classes. This is a repeated-measures comparison in which only prompt wording changes.

For $N$ samples, Top-1 accuracy is

$$
A=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}(\hat c_i=c_i).
$$

Runtime is secondary evidence because prompt strategies should be compared primarily on classification behavior, while computational cost helps document practical reproducibility.


## 8. Best Prompt and Class-wise Accuracy

### Data-Driven Selection

The retained prompt is the strategy with the highest measured Top-1 accuracy in the benchmark. It must not be selected from prior expectation.

For class $j$ with $N_j$ samples, class-wise accuracy is

$$
A_j=
\frac{1}{N_j}
\sum_{i:c_i=j}
\mathbf{1}(\hat c_i=c_i).
$$

This exposes categories that are hidden by the global mean.


## 9. Prompt Strategy Comparison

### Baseline Interpretation

The class-only template provides a minimal linguistic baseline. More descriptive prompts test whether contextual wording improves alignment between an audio embedding and the intended class prototype.

A prompt comparison should preserve the measured benchmark values rather than recomputing accuracy through an independent path. This keeps figures and tables traceable to one source of truth.


## 10. Class-wise Accuracy Summary

### Diagnostic Extremes

Lowest- and highest-performing classes are descriptive diagnostics. They can reveal acoustic overlap, semantic ambiguity, label phrasing effects, or strong pretrained representation for particular sound categories.

Extreme classes should be extracted from the complete class-wise table rather than evaluated separately, so the summary remains consistent with the global experiment.


## 11. Main Benchmark Summary

### Percentage-Point Improvement

For best accuracy $A_{best}$ and class-only baseline $A_{base}$,

$$
\Delta_{pp}
=
100(A_{best}-A_{base}).
$$

The unit is **percentage points**. This differs from relative percent improvement,

$$
100\frac{A_{best}-A_{base}}{A_{base}},
$$

which answers a different question and is not the primary quantity used in this study.


## 12. Top-k Accuracy

### Ranking Quality

Top-k accuracy counts a sample as correct when its true class appears among the $k$ highest similarity scores:

$$
A_k=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}
\left(
c_i\in \mathcal{R}_{i,k}
\right),
$$

where $\mathcal{R}_{i,k}$ is the set of the first $k$ ranked classes.

Because the candidate set only grows with $k$,

$$
A_1\le A_2\le\cdots\le A_{10}.
$$

This reveals whether Top-1 errors are close semantic alternatives or broader ranking failures.


## 13. AudioSet Metadata Analysis

### Association Analysis

Pearson correlation measures linear association between two numeric variables, while Spearman correlation measures monotonic association through ranks.

For paired variables $X$ and $Y$, Pearson correlation is

$$
r=
\frac{\operatorname{cov}(X,Y)}
{\sigma_X\sigma_Y}.
$$

Spearman correlation applies the same principle to ranked values.

### Interpretation Constraint

A weak or strong correlation does not establish causality. AudioSet quality and prevalence are only two possible explanatory variables among representation quality, acoustic distinctiveness, label semantics, and intra-class variability.


## 14. Comparison with Published ESC-50 Classifiers

### Protocol Comparability

Published ESC-50 systems may use supervised training, fine-tuning, augmentation, different pretraining corpora, or different evaluation procedures. Their reported accuracies therefore provide context rather than a controlled head-to-head experiment.

The project result is recomputed locally under a zero-shot protocol, while external values remain explicitly marked as literature references.

### Reference Methods

- **AST** — Audio Spectrogram Transformer, Gong, Chung & Glass (Interspeech 2021).
- **HTS-AT** — Hierarchical Token-Semantic Audio Transformer, Chen et al. (2022).
- **BEATs** — Audio Pre-Training with Acoustic Tokenizers, Chen et al. (ICML 2023).


## 15. Results Summary

### Traceable Reporting

The final summary is a deterministic reduction of objects already computed by the benchmark. This prevents divergence between narrative claims and executable results.

A robust summary should reuse the retained prompt, baseline, percentage-point gain, and Top-k dataframe rather than manually reproducing values in prose.


## Technical Synthesis

The project is a controlled inference study rather than a model-training study. The pretrained CLAP representation is fixed, ESC-50 defines the evaluation population, and prompt wording determines the text prototypes against which each audio embedding is compared.

The experimental logic is therefore:

```text
fixed audio sample
      ↓
CLAP audio embedding
      ↓
L2 normalization
      ↓
similarity against 50 text prototypes
      ↓
class ranking
      ↓
Top-1 / Top-k evaluation

prompt template + fixed class vocabulary
      ↓
CLAP text embeddings
      ↓
L2 normalization
```

The main scientific question is whether changing the linguistic context of otherwise identical class labels changes alignment enough to alter zero-shot classification performance.

## Scope and Limitations

### Included

- fixed pretrained LAION-CLAP inference;
- ten manually designed prompt templates;
- complete ESC-50 zero-shot evaluation;
- class-wise and Top-k analysis;
- AudioSet metadata association analysis;
- contextual comparison with published ESC-50 methods.

### Not included

- ESC-50-specific fine-tuning;
- learned or gradient-optimized prompts;
- prompt ensembling;
- checkpoint comparison;
- causal attribution of class performance to AudioSet metadata;
- controlled reproduction of the external supervised/fine-tuned benchmark systems.

Results are specific to the evaluated checkpoint, prompt family, class vocabulary, software environment, and dataset protocol. External benchmark values should be interpreted as context rather than direct experimental competitors.